<a href="https://colab.research.google.com/github/sharaffatima/bachelor-thesis/blob/main/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Imports
2. Paths
3. Load metadata
4. Convert scp_codes
5. Create binary labels
6. Split using strat_fold
7. Load ECG signals
8. Check shapes
9. Handle NaNs
10. Normalize using X_train only
11. Save processed arrays
12. Reload and verify

# 1. Paths
connect Github and Drive

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import os
import sys
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"



token = getpass("GitHub token: ")


REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
/content/bachelor-thesis
Already up to date.
/content/bachelor-thesis
Repo exists: True


In [4]:
! ls

explainability	notebooks  requirements.txt  thesis
models		README.md  tagebuch.md	     utils


In [5]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))

from utils.dataset import ECGReconstructionDataset #costume built dataset

print("Import worked.")

Import worked.


# 2. Paths to dataset

In [6]:
DATA_DIR = Path("/content/drive/MyDrive/ECG data")

RAW_DIR = DATA_DIR / "raw" / "ptbxl"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR exists:", RAW_DIR.exists())
print("ptbxl_database exists:", (RAW_DIR / "ptbxl_database.csv").exists())
print("scp_statements exists:", (RAW_DIR / "scp_statements.csv").exists())
print("records100 exists:", (RAW_DIR / "records100").exists())

RAW_DIR exists: True
ptbxl_database exists: True
scp_statements exists: True
records100 exists: True


# 3. Load Metadata of PTB XL

In [7]:
import pandas as pd
import ast

metadata = pd.read_csv(RAW_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp_statements = pd.read_csv(RAW_DIR / "scp_statements.csv", index_col=0)

metadata["scp_codes"] = metadata["scp_codes"].apply(ast.literal_eval)

metadata.head()

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


# 4. Labels
0 is normal

1 is anomaly

In [8]:
def is_normal(scp_codes):
    return int("NORM" in scp_codes)

metadata["is_normal"] = metadata["scp_codes"].apply(is_normal)
metadata["label"] = 1 - metadata["is_normal"]

metadata["label"].value_counts()

label
1    12285
0     9514
Name: count, dtype: int64

# 5. Split
Now I create the Split based on the official PTB XL recommended folds: 8 Traingn, 1 Validation, 1 Testing

For now, I'll only take 1000 samples

In [16]:
RANDOM_SEED = 42

train_meta_all = metadata[metadata["strat_fold"].between(1, 8)]
val_meta_all = metadata[metadata["strat_fold"] == 9]
test_meta_all = metadata[metadata["strat_fold"] == 10]

train_normal_all = train_meta_all[train_meta_all["label"] == 0]

# Small subset sizes
N_TRAIN = 600
N_VAL = 200
N_TEST = 200

train_meta = train_normal_all.sample(
    n=N_TRAIN,
    random_state=RANDOM_SEED
)

val_meta = val_meta_all.sample(
    n=N_VAL,
    random_state=RANDOM_SEED
)

test_meta = test_meta_all.sample(
    n=N_TEST,
    random_state=RANDOM_SEED
)

In [17]:
print("Train:")
print(train_meta["label"].value_counts())

print("\nValidation:")
print(val_meta["label"].value_counts())

print("\nTest:")
print(test_meta["label"].value_counts())

Train:
label
0    600
Name: count, dtype: int64

Validation:
label
1    108
0     92
Name: count, dtype: int64

Test:
label
1    114
0     86
Name: count, dtype: int64


Load One signal to test

In [11]:
! pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 777.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 36.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


In [10]:
import wfdb

sample_file = metadata.iloc[0]["filename_lr"]
record = wfdb.rdrecord(str(RAW_DIR / sample_file))
signal = record.p_signal

print(signal.shape)

(1000, 12)


# 6. Load all Data

only 1000 samples for now

In [11]:
import numpy as np
from tqdm import tqdm

def load_signals(meta_df, raw_dir):
    signals = []

    for filename in tqdm(meta_df["filename_lr"]):
        record = wfdb.rdrecord(str(raw_dir / filename))
        signals.append(record.p_signal)

    return np.stack(signals).astype(np.float32)

In [18]:
X_train = load_signals(train_meta, RAW_DIR)
X_val = load_signals(val_meta, RAW_DIR)
X_test = load_signals(test_meta, RAW_DIR)

y_val = val_meta["label"].to_numpy().astype(np.int64)
y_test = test_meta["label"].to_numpy().astype(np.int64)

100%|██████████| 200/200 [04:12<00:00,  1.26s/it]


# 7. Check shapes



In [19]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (600, 1000, 12)
X_val: (200, 1000, 12)
X_test: (200, 1000, 12)
y_val: (200,)
y_test: (200,)


# 7. Handle Nans

In [20]:
X_train = np.nan_to_num(X_train, nan=0.0)
X_val = np.nan_to_num(X_val, nan=0.0)
X_test = np.nan_to_num(X_test, nan=0.0)

# 8. Normalize

In [21]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train_norm = (X_train - mean) / (std + 1e-8)
X_val_norm = (X_val - mean) / (std + 1e-8)
X_test_norm = (X_test - mean) / (std + 1e-8)

# 9. Save processes data

In [22]:
np.save(PROCESSED_DIR / "X_train_small.npy", X_train_norm)
np.save(PROCESSED_DIR / "X_val_small.npy", X_val_norm)
np.save(PROCESSED_DIR / "X_test_small.npy", X_test_norm)

np.save(PROCESSED_DIR / "y_val_small.npy", y_val)
np.save(PROCESSED_DIR / "y_test_small.npy", y_test)

np.save(PROCESSED_DIR / "train_ids_small.npy", train_meta.index.to_numpy())
np.save(PROCESSED_DIR / "val_ids_small.npy", val_meta.index.to_numpy())
np.save(PROCESSED_DIR / "test_ids_small.npy", test_meta.index.to_numpy())

np.savez(
    PROCESSED_DIR / "normalization_stats_small.npz",
    mean=mean,
    std=std
)